In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import gdown

### Question 1: [IPO] Withdrawn IPOs by Company Type
#### What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?

In [2]:
url = "https://www.iposcoop.com/ipos-recently-filed/"

In [3]:
tables = pd.read_html(url)

In [4]:
ipo_df = tables[0]

In [9]:
ipo_df.shape

(500, 10)

In [10]:
ipo_df.columns.tolist()

['File Date',
 'Company',
 'Symbol',
 'Managers',
 'Shares (millions)',
 'Price Low',
 'Price High',
 'Est $ Vol (millions)',
 'Expected To Trade',
 'SCOOP Rating']

In [11]:
ipo_df.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
4,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Week of,S/O


In [12]:
withdrawn_df = ipo_df[ipo_df["Expected To Trade"] == "Withdrawn"].copy()

In [13]:
withdrawn_df["File Date"] = pd.to_datetime(withdrawn_df["File Date"])

In [14]:
withdrawn_df = withdrawn_df[withdrawn_df["File Date"] < "2026-09-11"].copy()

In [15]:
len(withdrawn_df)

31

In [16]:
withdrawn_df[["File Date", "Company", "Expected To Trade"]].head(10)

,File Date,Company,Expected To Trade
38,2026-09-10,Motive Technologies (Withdrawn),Withdrawn
43,2026-09-04,Idea Tech Holding (Withdrawn),Withdrawn
50,2026-09-01,Hornbeck Offshore Services (Withdrawn),Withdrawn
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),Withdrawn
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),Withdrawn
69,2026-08-24,pan-Africa Corp. (Withdrawn),Withdrawn
84,2026-08-18,Living Homeopathy (Withdrawn),Withdrawn
128,2026-07-31,Nuclea Energy (Withdrawn),Withdrawn
158,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),Withdrawn
163,2026-07-08,Cloud Data Holdings (Withdrawn),Withdrawn


In [17]:
def get_company_type(company_name):
    name = str(company_name)

    if "Technologies" in name:
        return "Technologies"
    if "Acquisition Corp" in name or "Acquisition Corporation" in name or "Corp" in name:
        return "Acquisition Corp"
    if "Inc" in name or "Incorporated" in name:
        return "Inc."
    if "Group" in name:
        return "Group"
    if "Ltd" in name or "Limited" in name:
        return "Limited"
    if "Holdings" in name or "Holding" in name:
        return "Holdings"
    return "Other"

In [18]:
withdrawn_df["Company Type"] = withdrawn_df["Company"].apply(get_company_type)

In [19]:
withdrawn_df["Company Type"].value_counts()

,count
Company Type,
Other,8
Limited,7
Acquisition Corp,5
Holdings,4
Technologies,3
Group,3
Inc.,1


In [20]:
withdrawn_df[["Company", "Company Type"]].head(15)

,Company,Company Type
38,Motive Technologies (Withdrawn),Technologies
43,Idea Tech Holding (Withdrawn),Holdings
50,Hornbeck Offshore Services (Withdrawn),Other
52,Coolbit Technologies Ltd. (Withdrawn),Technologies
59,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp
69,pan-Africa Corp. (Withdrawn),Acquisition Corp
84,Living Homeopathy (Withdrawn),Other
128,Nuclea Energy (Withdrawn),Other
158,APEX Global Solutions Ltd. (WITHDRAWN),Limited
163,Cloud Data Holdings (Withdrawn),Holdings


In [21]:
def parse_price(value):
    if pd.isna(value) or value in ["-", "TBA", ""]:
        return np.nan
    try:
        clean = str(value).replace("$", "").replace(",", "").strip()
        return float(clean)
    except:
        return np.nan

In [22]:
withdrawn_df["Price_Low_num"] = withdrawn_df["Price Low"].apply(parse_price)
withdrawn_df["Price_High_num"] = withdrawn_df["Price High"].apply(parse_price)
withdrawn_df["Avg_price"] = (withdrawn_df["Price_Low_num"] + withdrawn_df["Price_High_num"]) / 2

In [23]:
withdrawn_df[["Price Low", "Price High", "Avg_price"]].head(10)

,Price Low,Price High,Avg_price
38,NaN,NaN,NaN
43,$4.00,$5.00,4.50
50,NaN,NaN,NaN
52,$4.00,$5.00,4.50
59,$10.00,$10.00,10.00
69,$10.00,$10.00,10.00
84,$4.00,$6.00,5.00
128,$8.00,$10.00,9.00
158,$4.00,$4.00,4.00
163,$4.00,$4.50,4.25


In [24]:
def to_numeric_clean(value):
    if pd.isna(value) or value in ["-", "TBA", ""]:
        return np.nan
    try:
        clean = str(value).replace("$", "").replace(",", "").strip()
        return float(clean)
    except:
        return np.nan

In [25]:
withdrawn_df["Shares_num"] = withdrawn_df["Shares (millions)"].apply(to_numeric_clean)
withdrawn_df["Est_vol_num"] = withdrawn_df["Est $ Vol (millions)"].apply(to_numeric_clean)

In [26]:
withdrawn_df[["Shares (millions)", "Est $ Vol (millions)", "Shares_num", "Est_vol_num"]].head()

,Shares (millions),Est $ Vol (millions),Shares_num,Est_vol_num
38,0.0,$100.00,0.0,100.0
43,2.0,$9.00,2.0,9.0
50,0.0,$100.00,0.0,100.0
52,5.0,$22.50,5.0,22.5
59,20.0,$200.00,20.0,200.0


In [27]:
shares_times_price = withdrawn_df["Shares_num"] * withdrawn_df["Avg_price"]
withdrawn_df["Shares_offered_value"] = shares_times_price.fillna(withdrawn_df["Est_vol_num"])

In [28]:
withdrawn_df[["Company", "Shares_num", "Avg_price", "Est_vol_num", "Shares_offered_value"]].head(10)

,Company,Shares_num,Avg_price,Est_vol_num,Shares_offered_value
38,Motive Technologies (Withdrawn),0.00,NaN,100.00,100.0000
43,Idea Tech Holding (Withdrawn),2.00,4.50,9.00,9.0000
50,Hornbeck Offshore Services (Withdrawn),0.00,NaN,100.00,100.0000
52,Coolbit Technologies Ltd. (Withdrawn),5.00,4.50,22.50,22.5000
59,Timber Road Acquisition Corp. (Withdrawn),20.00,10.00,200.00,200.0000
69,pan-Africa Corp. (Withdrawn),12.50,10.00,125.00,125.0000
84,Living Homeopathy (Withdrawn),3.75,5.00,19.00,18.7500
128,Nuclea Energy (Withdrawn),5.60,9.00,50.40,50.4000
158,APEX Global Solutions Ltd. (WITHDRAWN),3.75,4.00,15.00,15.0000
163,Cloud Data Holdings (Withdrawn),3.75,4.25,15.94,15.9375


In [29]:
type_totals = withdrawn_df.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)

In [30]:
type_totals

,Shares_offered_value
Company Type,
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Other,290.4450
Limited,197.2500
Technologies,184.9000
Group,32.5000


In [31]:
best_type = type_totals.index[0]
best_value = type_totals.iloc[0]

In [32]:
print(f"Company type with highest withdrawn value: {best_type}")

Company type with highest withdrawn value: Acquisition Corp


In [33]:
print(f"Total withdrawn value: {best_value:.2f} $ millions")

Total withdrawn value: 499.99 $ millions


### Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)
#### What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?

In [87]:
url_2025 = "https://www.iposcoop.com/2025-pricings/"

In [88]:
tables_2025 = pd.read_html(url_2025)

In [89]:
ipo_2025 = tables_2025[0]

In [90]:
len(ipo_2025)

231

In [91]:
ipo_2025.columns.tolist()

['Company',
 'Symbol',
 'Industry',
 'Offer Date',
 'Shares (millions)',
 'Offer Price',
 '1st Day Close',
 'Current Price',
 'Return',
 'SCOOP Rating']

In [92]:
ipo_2025.head()

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


In [93]:
ipo_2025["Offer Date"] = pd.to_datetime(ipo_2025["Offer Date"])

In [94]:
mask_date = ipo_2025["Offer Date"] < "2025-09-01"
mask_return = ipo_2025["Return"] != "0.00%"

In [95]:
ipo_filtered = ipo_2025[mask_date & mask_return].copy()

In [96]:
len(ipo_filtered)

146

In [97]:
ticker_list = ipo_filtered["Symbol"].tolist()

In [98]:
ticker_list[:10]

['PMI', 'TGHL', 'THH', 'CURX', 'CEPF', 'ETS', 'YMT', 'NUTR', 'PPCB', 'BUUU']

In [101]:
raw_data = yf.download(
    ticker_list,
    start="2025-01-01",
    end="2026-09-12",
    group_by="ticker",
    auto_adjust=True,
    progress=True,
    threads=True
)

[*********************100%***********************]  146 of 146 completed
ERROR:yfinance:
14 Failed downloads:
ERROR:yfinance:['PTNM', 'AHL', 'TBH', 'SKBL', 'MCTR', 'CAEP', 'MJID', 'CEPT', 'SDM', 'WGRX', 'EMPG', 'MTSR', 'AGH', 'EPWK']: YFTzMissingError('possibly delisted; no timezone found')


In [102]:
raw_data.shape

(424, 744)

In [103]:
stocks_list = []

In [104]:
for ticker in ticker_list:
    try:
        if ticker not in raw_data.columns.get_level_values(0):
            continue
        ticker_data = raw_data[ticker].copy()
        ticker_data = ticker_data.dropna(how="all")
        if len(ticker_data) == 0:
            continue
        ticker_data["Ticker"] = ticker
        ticker_data = ticker_data.reset_index()
        stocks_list.append(ticker_data)
    except Exception as e:
        continue

In [105]:
if len(stocks_list) > 0:
    stocks_df = pd.concat(stocks_list, ignore_index=True)
else:
    stocks_df = pd.DataFrame()

In [106]:
stocks_df["Ticker"].nunique() if len(stocks_df) > 0 else 0

132

In [107]:
len(stocks_df)

44821

In [108]:
stocks_df.head()

Price,Date,Open,High,Low,Close,Volume,Ticker
0,2025-08-29,210.5,356.0,209.5,229.5,126794.0,PMI
1,2025-09-02,242.5,267.5,232.5,257.0,98358.0,PMI
2,2025-09-03,257.0,270.0,250.5,264.5,35960.0,PMI
3,2025-09-04,265.5,287.5,265.0,286.5,28074.0,PMI
4,2025-09-05,288.0,295.0,282.5,292.5,18572.0,PMI


In [109]:
stocks_df = stocks_df.sort_values(["Ticker", "Date"])

In [110]:
stocks_df["growth_252d"] = stocks_df.groupby("Ticker")["Close"].transform(
    lambda x: x / x.shift(252)
)

In [111]:
stocks_df["volatility"] = stocks_df.groupby("Ticker")["Close"].transform(
    lambda x: x.rolling(30).std() * np.sqrt(252)
)

In [112]:
stocks_df[["Date", "Ticker", "Close", "growth_252d", "volatility"]].dropna().head()

Price,Date,Ticker,Close,growth_252d,volatility
39661,2026-01-27,AAPG,25.590000,1.472382,19.970525
39662,2026-01-28,AAPG,25.299999,1.466667,18.766079
39663,2026-01-29,AAPG,25.000000,1.453488,19.781113
39664,2026-01-30,AAPG,24.180000,1.395268,20.297329
39665,2026-02-02,AAPG,23.170000,1.316477,22.505299


In [113]:
rf_rate = 0.05
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - rf_rate) / stocks_df["volatility"]

In [114]:
stocks_df[["Date", "Ticker", "growth_252d", "volatility", "Sharpe"]].dropna().head()

Price,Date,Ticker,growth_252d,volatility,Sharpe
39661,2026-01-27,AAPG,1.472382,19.970525,0.071224
39662,2026-01-28,AAPG,1.466667,18.766079,0.075491
39663,2026-01-29,AAPG,1.453488,19.781113,0.070951
39664,2026-01-30,AAPG,1.395268,20.297329,0.066278
39665,2026-02-02,AAPG,1.316477,22.505299,0.056275


In [115]:
target_date = pd.Timestamp("2026-09-11")

In [116]:
snap = stocks_df[stocks_df["Date"] == target_date].copy()

In [117]:
len(snap)

132

In [118]:
snap[["growth_252d", "volatility", "Sharpe"]].describe()

Price,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,27.516340,inf
std,3.049007,66.879183,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,627.698106,inf


In [119]:
median_sharpe = snap["Sharpe"].median()

In [120]:
print(f"Median Sharpe ratio as of 2026-09-11: {median_sharpe:.4f}")

Median Sharpe ratio as of 2026-09-11: 0.0501


### Question 3: [IPO] 'Fixed Months Holding Strategy'
#### What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?

In [121]:
for m in range(1, 13):
    days = m * 21
    col_name = f"future_growth_{m}_m"
    stocks_df[col_name] = stocks_df.groupby("Ticker")["Close"].transform(
        lambda x: x.shift(-days) / x
    )

In [122]:
[c for c in stocks_df.columns if c.startswith("future_growth")]

['future_growth_1_m',
 'future_growth_2_m',
 'future_growth_3_m',
 'future_growth_4_m',
 'future_growth_5_m',
 'future_growth_6_m',
 'future_growth_7_m',
 'future_growth_8_m',
 'future_growth_9_m',
 'future_growth_10_m',
 'future_growth_11_m',
 'future_growth_12_m']

In [123]:
first_days = stocks_df.groupby("Ticker")["Date"].min().reset_index()
first_days.columns = ["Ticker", "min_date"]

In [124]:
first_close = stocks_df.merge(first_days, left_on=["Ticker", "Date"], right_on=["Ticker", "min_date"])
first_close = first_close[["Ticker", "min_date", "Close"]].rename(columns={"Close": "ipo_close"})

In [125]:
len(first_close)

132

In [126]:
first_close.head()

,Ticker,min_date,ipo_close
0,AAPG,2025-01-24,17.379999
1,AARD,2025-02-13,14.310000
2,ADVB,2025-03-06,73.000000
3,AII,2025-05-08,16.008226
4,AIRO,2025-06-13,24.000000


In [127]:
ipo_day_rows = stocks_df.merge(first_days, left_on=["Ticker", "Date"], right_on=["Ticker", "min_date"])

In [128]:
growth_cols = [f"future_growth_{m}_m" for m in range(1, 13)]
ipo_growth = ipo_day_rows[["Ticker", "min_date"] + growth_cols].copy()

In [129]:
ipo_growth.shape

(132, 14)

In [130]:
ipo_growth.head()

,Ticker,min_date,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
0,AAPG,2025-01-24,1.159954,1.056962,1.512946,1.485328,2.348101,2.259494,2.459436,2.176640,2.040276,1.842923,1.604143,1.472382
1,AARD,2025-02-13,0.647100,0.555556,0.635220,0.758211,0.835779,0.753319,0.716981,0.973445,0.758910,1.011181,1.072676,0.870021
2,ADVB,2025-03-06,0.920548,0.523288,0.255069,0.181370,0.099178,0.114521,0.132329,0.124932,0.104110,0.095890,0.065479,0.067808
3,AII,2025-05-08,1.005917,1.020118,1.039645,1.156213,1.414793,1.469231,1.156213,1.196450,1.049112,1.124852,1.139414,1.236239
4,AIRO,2025-06-13,1.085417,0.955833,0.841667,0.836667,0.557083,0.354167,0.551250,0.383333,0.442500,0.360417,0.265417,0.329167


In [131]:
ipo_growth[growth_cols].describe()

,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,130.000000
mean,95.751832,67.979575,51.249160,22.699034,54.244577,64.013104,59.268089,21.460472,20.010294,11.260278,8.904178,5.823784
std,1087.893336,764.388639,573.269360,247.825677,609.321778,720.716187,666.109314,234.733129,216.163486,115.714394,90.034523,55.496952
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.710433,0.584863,0.442477,0.392759,0.413434,0.308026,0.245548,0.196227,0.175276,0.174333,0.159666,0.156935
50%,0.935351,0.892958,0.827160,0.730400,0.690909,0.726000,0.662000,0.603508,0.580294,0.533333,0.481811,0.491838
75%,1.114199,1.164977,1.129961,1.197492,1.077133,1.041222,1.018281,1.036588,1.031006,1.034765,1.106045,1.041541
max,12500.000279,8750.000196,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


In [132]:
medians = ipo_growth[growth_cols].median()
best_month_col = medians.idxmax()
best_month = int(best_month_col.split("_")[2])
best_median = medians.max()

In [133]:
print(f"Optimal holding period: {best_month} months")

Optimal holding period: 1 months


In [134]:
print(f"Maximum median growth: {best_median:.4f}")

Maximum median growth: 0.9354


### Question 4: [Strategy] Simple RSI-Based Trading Strategy
#### What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?

In [135]:
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"

In [136]:
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=9c78a48d-fd38-43af-bd9a-2e663f36ba33
To: /content/data.parquet
100%|██████████| 130M/130M [00:01<00:00, 84.5MB/s]


'data.parquet'

In [137]:
df = pd.read_parquet(
    "data.parquet",
    engine="pyarrow",
    columns=["Date", "Ticker", "rsi", "growth_future_30d"]
)

In [138]:
df.shape

(229932, 4)

In [139]:
df.head()

,Date,Ticker,rsi,growth_future_30d
0,1986-03-13,MSFT,NaN,0.982144
1,1986-03-14,MSFT,NaN,0.922418
2,1986-03-17,MSFT,NaN,0.881354
3,1986-03-18,MSFT,NaN,0.921744
4,1986-03-19,MSFT,NaN,0.964604


In [140]:
print("Date range:", df["Date"].min(), "->", df["Date"].max())

Date range: 1972-06-01 00:00:00 -> 2025-05-30 00:00:00


In [141]:
rsi_threshold = 30

In [142]:
mask = (
    (df["Date"] >= "2000-01-01") &
    (df["Date"] <= "2025-06-01") &
    (df["rsi"] < rsi_threshold)
)

In [143]:
selected_df = df[mask].copy()

In [144]:
len(selected_df)

5206

In [145]:
net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()

In [146]:
selected_df["growth_future_30d"].mean()

np.float64(1.0126403360751977)

In [148]:
(selected_df["growth_future_30d"] > 1).mean()

np.float64(0.5512869765655013)

In [149]:
print(f"Net income: {net_income:.2f} dollars")

Net income: 65805.59 dollars


In [150]:
print(f"Net income in $ thousands: {net_income / 1000:.2f}")

Net income in $ thousands: 65.81
